# 02 — Baseline & Reward Rates

**Goal:** Compute the random-policy baseline (the number to beat) and raw per-template reward rates.

This notebook answers:
- What is the overall reward rate under random template selection?
- Which templates have the highest/lowest raw reward rates?
- Why can raw rates be misleading? (motivation for RDS)

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_sample, parse_eligible_templates
from src.evaluation.baseline import compute_random_baseline
from src.scoring.difference_score import compute_template_reward_rates

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

print("Ready!")

---
## 1. Load Training Data (100K sample)

In [ ]:
train_df = load_sample(n_rows=100_000, split="train")
print(f"\nShape: {train_df.shape}")
train_df.head()

---
## 2. Random Baseline — The Number to Beat

The dataset was collected under a **random policy** — Duolingo picked each template uniformly at random from the eligible set. So the average reward in the data equals the random policy's performance.

Everything we build later must beat this number. If it doesn't, our algorithm is worse than flipping a coin.

In [ ]:
baseline = compute_random_baseline(train_df)

print(f"\n{'='*40}")
print(f"  RANDOM BASELINE: {baseline:.6f} ({baseline:.2%})")
print(f"{'='*40}")
print(f"\nThis means: under random template selection,")
print(f"about {baseline:.1%} of users complete a lesson within 2 hours.")
print(f"\nOur goal: beat {baseline:.4f} using smarter template choices.")

---
## 3. Per-Template Reward Rates

For each template A–L, we compute:

$$\text{reward\_rate}(a) = \frac{\text{# times template a was sent AND user engaged}}{\text{# times template a was sent}}$$

This is the simplest way to rank templates — but as we'll see, it has a serious flaw.

In [ ]:
reward_rates, counts = compute_template_reward_rates(train_df)

In [ ]:
# Build a summary dataframe
summary = pd.DataFrame({
    "template": list(reward_rates.keys()),
    "reward_rate": list(reward_rates.values()),
    "count": [counts[t] for t in reward_rates.keys()]
}).sort_values("reward_rate", ascending=False).reset_index(drop=True)

summary["rank"] = range(1, len(summary) + 1)
print("Template ranking by raw reward rate:\n")
print(summary[["rank", "template", "reward_rate", "count"]].to_string(index=False))

In [ ]:
# Bar chart of reward rates
fig, ax = plt.subplots(figsize=(10, 5))

colors = ["#2ecc71" if r > baseline else "#e74c3c" for r in summary["reward_rate"]]

bars = ax.bar(summary["template"], summary["reward_rate"], color=colors, edgecolor="white")
ax.axhline(y=baseline, color="navy", linestyle="--", linewidth=1.5,
           label=f"Random baseline: {baseline:.4f}")

ax.set_title("Raw Reward Rate per Template", fontsize=14, fontweight="bold")
ax.set_xlabel("Template")
ax.set_ylabel("Reward Rate")
ax.legend(fontsize=11)

# Annotate bars
for bar, rate in zip(bars, summary["reward_rate"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{rate:.4f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# How many events per template? (should be ~uniform since random policy)
fig, ax = plt.subplots(figsize=(10, 4))

count_df = summary.sort_values("template")
ax.bar(count_df["template"], count_df["count"], color="steelblue", edgecolor="white")
ax.axhline(y=count_df["count"].mean(), color="red", linestyle="--",
           label=f"Mean: {count_df['count'].mean():,.0f}")

ax.set_title("Number of Events per Template (should be roughly uniform)", fontsize=13)
ax.set_xlabel("Template")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Count range: {count_df['count'].min():,} – {count_df['count'].max():,}")
print(f"Note: counts are NOT perfectly equal because eligibility varies per event.")

---
## 4. Why Raw Reward Rates Can Be Misleading

The raw reward rate treats every event equally. But there's an important confounding factor:

**Different templates are eligible for different users at different times.**

Imagine template A is mostly eligible for **active users** (who engage with anything), while template B is mostly eligible for **dormant users** (who rarely engage). Template A's raw reward rate will look higher — but that's because of the *users*, not the *template*.

Let's check whether the eligible set composition actually varies across templates.

In [ ]:
# For each template, what's the average reward when it IS vs ISN'T in the eligible set?
# This shows whether eligibility correlates with user engagement.

templates = sorted(reward_rates.keys())

eligible_stats = []
for t in templates:
    is_eligible = train_df["eligible_templates"].apply(
        lambda x: t in (x if isinstance(x, list) else parse_eligible_templates(x))
    )
    reward_when_eligible = train_df.loc[is_eligible, "session_end_completed"].mean()
    reward_when_not = train_df.loc[~is_eligible, "session_end_completed"].mean()
    pct_eligible = is_eligible.mean()
    eligible_stats.append({
        "template": t,
        "pct_eligible": pct_eligible,
        "reward_when_eligible": reward_when_eligible,
        "reward_when_not_eligible": reward_when_not,
        "gap": reward_when_eligible - reward_when_not if pd.notna(reward_when_not) else 0
    })

elig_df = pd.DataFrame(eligible_stats)
print("Reward rate when template IS vs IS NOT in the eligible set:\n")
print(elig_df.to_string(index=False))

print(f"\nIf all gaps were zero, eligibility wouldn't matter.")
print(f"Non-zero gaps = eligibility is confounded with user engagement.")
print(f"→ This is exactly why we need Relative Difference Scores (next notebook).")

---
## 5. Summary

| What we computed | Value |
|-----------------|-------|
| **Random baseline** | Overall mean reward rate — the number to beat |
| **Per-template reward rates** | Raw success rate for each template A–L |
| **Template ranking** | Which templates look best *naively* |
| **Confounding evidence** | Eligibility composition varies → raw rates are biased |

**Key takeaway:** Raw reward rates are a useful first look, but they mix up template quality with user quality. A template might look great just because it's eligible for active users.

**Next notebook:** `03_rds_scoring.ipynb` — compute **Relative Difference Scores** that separate template quality from user quality.